In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import chi2_contingency, ttest_ind
import seaborn as sns
import matplotlib.pyplot as plt

#PULIZIA DATASET
df = pd.read_csv("/Users/liadraetta/Desktop/progetti/Projects/tutorial-experiment/directors_full.csv")

variables = ['population', 'gdpNominal']
df[variables] = df[variables].replace(0, np.nan)
df = df.dropna(subset=variables)  # Rimuovi righe con NaN

DISTRIBUTION PER COUNTRY POPULATIONS

In [ ]:

bins = [0, 1_000_000, 10_000_000, 100_000_000, 1_000_000_000, np.inf]
labels = ['Microstates','Lower-Middle Population','Upper-Middle Population','Large States','Demographic Giants']

# Applicazione della logica con pd.cut
df['population_bin'] = pd.cut(
    df['population'], 
    bins=bins, 
    labels=labels, 
    right=False #
)

head_tail_counts = df.groupby(['population_bin', 'type'], observed=False).size().unstack(fill_value=0)

print(head_tail_counts)

In [ ]:
# T-test
for var in ["population"]:
    head_values = df[df['type'] == 'Head'][var].dropna()
    tail_values = df[df['type'] == 'Long-tail'][var].dropna()
    
    t_stat, p_value = ttest_ind(head_values, tail_values)
    print(f"{var}: t={t_stat:.3f}, p={p_value:.3f}")

In [ ]:
for var in ["population"]:
    plt.figure(figsize=(6,4))
    sns.boxplot(x='type', y=var, data=df)
    plt.title(f'{var} per tipo')
    plt.show()

DISTRIBUTION PER GDP

In [ ]:
# T-test
for var in ["gdpNominal"]:
    head_values = df[df['type'] == 'Head'][var].dropna()
    tail_values = df[df['type'] == 'Long-tail'][var].dropna()
    
    t_stat, p_value = ttest_ind(head_values, tail_values)
    print(f"{var}: t={t_stat:.3f}, p={p_value:.3f}")

In [ ]:
for var in ["gdpNominal"]:
    plt.figure(figsize=(6,4))
    sns.boxplot(x='type', y=var, data=df)
    plt.title(f'{var} per tipo')
    plt.show()

DISTRIBUTION PER GENDER

In [ ]:
gender_counts = df.groupby(['norm_gender', 'type']).size().unstack(fill_value=0)
print(gender_counts)

In [ ]:
# Test chi-quadro per gender
for cat_var in ['norm_gender']:
    contingency_table = pd.crosstab(df[cat_var], df['type'])
    chi2, p, dof, expected = chi2_contingency(contingency_table)
    print(f"{cat_var}: chi2={chi2:.3f}, p={p:.3f}")

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Tabella contingenza
contingency = pd.crosstab(df['norm_gender'], df['type'])
print(contingency)

# Grafico
sns.countplot(x='norm_gender', hue='type', data=df)
plt.title('Distribuzione Head/Tail per norm_gender')
plt.show()

DISTRIBUZION PER unm49

In [ ]:
gender_counts = df.groupby(['unm49', 'type']).size().unstack(fill_value=0)
print(gender_counts)

In [ ]:
for cat_var in ['unm49']:
    contingency_table = pd.crosstab(df[cat_var], df['type'])
    chi2, p, dof, expected = chi2_contingency(contingency_table)
    print(f"{cat_var}: chi2={chi2:.3f}, p={p:.3f}")

In [ ]:
sns.countplot(x='unm49', hue='type', data=df)
plt.title('Distribuzione Head/Tail per norm_gender')
plt.show()

In [ ]:
import plotly.express as px

df['gdpNominal'] = pd.to_numeric(df['gdpNominal'], errors='coerce')
fig = px.scatter(
    df, 
    x="claims", 
    y="gdpNominal",
    color="type",       # Colors the dots by gender
    hover_name="label_en",     # Shows the name when you hover
    hover_data=["citizenship"],# Adds extra info to the hover box
    trendline="ols",           # Adds the trendline similar to your image
    title="Relationship between Number of Claims and Nominal GDP",
    labels={
        "claims": "Number of Claims",
        "gdpNominal": "GDP Nominal (USD)"
    }
)

# 4. Improve the layout
fig.update_layout(
    template="plotly_white",
    xaxis_title="Number of Claims",
    yaxis_title="Nominal GDP"
)

# 5. Show the plot
fig.show()

In [ ]:
import pandas as pd
from scipy.stats import kruskal

# Caricare il dataset
df = pd.read_csv("/Users/liadraetta/Desktop/progetti/Projects/tutorial-experiment/directors_full.csv", index_col=0)

# Creare una lista dei gruppi di claims in base a unm49
groups = [group["claims"].values for name, group in df.groupby("unm49")]

# Applicare il test di Kruskal-Wallis
stat, p_value = kruskal(*groups)

print(f"Kruskal-Wallis statistic: {stat}")
print(f"P-value: {p_value}")

if p_value < 0.05:
    print("La differenza tra le medie dei gruppi è significativa.")
else:
    print("La differenza tra le medie dei gruppi NON è significativa.")